<a href="https://colab.research.google.com/github/sylc4b7/outpeform/blob/release/Outperform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Open Google Sheet](https://docs.google.com/spreadsheets/d/1dDpRrnEs7otDwbr_746WP6nz8_UWOI2wVGzBISsHOUE/edit?usp=sharing)


In [6]:
import yfinance as yf
import pandas as pd
import requests
import json
import datetime
import gspread
from io import StringIO
from datetime import datetime, timedelta
from google.colab import auth
from gspread_dataframe import set_with_dataframe
from google.auth import default
import contextlib
import sys

# Authenticate and create the interface to Sheets
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Open the Google Sheet using its name or URL
spreadsheet = gc.open('Outperform')
worksheet = spreadsheet.worksheet('Stockcode')

# Assuming the specific column value you're looking for is "^hsi"
target_value = "123"

# Initialize an empty list to store stock symbols
stocks = []

# Iterate over each stock symbol
for stock in worksheet.col_values(1)[1:]:
    # Check if the current stock symbol matches the target value
    if stock == target_value:
        break  # Stop collecting symbols once the target value is found
    else:
        stocks.append(stock)  # Add the stock symbol to the list

def fetch_stock_data(ticker, start_date, end_date):
    # Suppress the console output
    with contextlib.redirect_stdout(None):
        # Fetch historical stock data from Yahoo Finance
        stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
    return stock_data

def calculate_happiness_lines(stock_data, window=20):
    # Calculate the mean and standard deviations
    with contextlib.redirect_stdout(None):
        stock_data['Mean'] = stock_data['Close'].rolling(window=window).mean()
        stock_data['StdDev'] = stock_data['Close'].rolling(window=window).std()

        # Calculate the five lines of happiness
        stock_data['Upper Band'] = stock_data['Mean'] + 2 * stock_data['StdDev']  # Mean + 2 StdDev
        stock_data['Lower Band'] = stock_data['Mean'] - 2 * stock_data['StdDev']  # Mean - 2 StdDev

    return stock_data

def get_roa_roe(ticker):
    # Fetch the stock data
    stock = yf.Ticker(ticker)

    # Get the financials data
    financials = stock.financials
    balance_sheet = stock.balance_sheet

    # Calculate ROA and ROE
    try:
        net_income = financials.loc['Net Income'].iloc[0]
        total_assets = balance_sheet.loc['Total Assets'].iloc[0]
        total_equity = balance_sheet.loc['Stockholders Equity'].iloc[0]

        roa = net_income / total_assets
        roe = net_income / total_equity

        return roa, roe
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        return None, None

def main():
    # Calculate the current date and the date 500 days before the current date
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=500)).strftime('%Y-%m-%d')

    print("\n\n" + datetime.now().strftime('%Y-%m-%d') + "\n\n")

    # Initialize an empty DataFrame to concatenate results
    concatenated_df = pd.DataFrame()

    for ticker in stocks:
        # Fetch data
        stock_data = fetch_stock_data(ticker, start_date, end_date)

        # Calculate happiness lines with a 20-day rolling window
        result = calculate_happiness_lines(stock_data, window=200)

        # Get the last row of the result
        last_row = result[['Close', 'Mean', 'Upper Band', 'Lower Band']].dropna().tail(1)

        # Convert the last row to an array of values
        last_row_values = last_row.values.flatten()

        # Create a DataFrame from the array of values
        last_row_df = pd.DataFrame([last_row_values], columns=['Close', 'Mean', 'Upper Band', 'Lower Band'])

        # Add the ticker symbol to the DataFrame
        last_row_df.insert(0, 'Ticker', ticker)

        # Get ROA and ROE
        roa, roe = get_roa_roe(ticker)
        last_row_df['ROA'] = roa
        last_row_df['ROE'] = roe

        # Concatenate the last row DataFrame to the concatenated DataFrame
        concatenated_df = pd.concat([concatenated_df, last_row_df], ignore_index=True)

    # Print the concatenated DataFrame
    #print(datetime.now().strftime('%Y-%m-%d'))
    print(concatenated_df)

     # Open the 'Current5' worksheet
    worksheet = spreadsheet.worksheet('Yahoo')

    # Set the DataFrame to the worksheet
    set_with_dataframe(worksheet, concatenated_df)

if __name__ == "__main__":
    main()



2024-12-05


Error fetching data for ^hsi: 'Net Income'


<ipython-input-6-47bb5cc89b42>:114: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  concatenated_df = pd.concat([concatenated_df, last_row_df], ignore_index=True)


Error fetching data for ^GSPC: 'Net Income'
     Ticker         Close          Mean    Upper Band    Lower Band       ROA  \
0   6605.tw    237.500000    225.725000    288.599586    162.850414  0.074292   
1     GOOGL    174.369995    164.895300    190.498462    139.292138  0.183391   
2         V    309.899994    278.911101    305.548675    252.273527  0.208896   
3      NFLX    911.059998    673.854950    830.729264    516.980635  0.110974   
4      NICE    192.389999    193.634950    255.103358    132.166541  0.066105   
5        SQ     98.919998     71.372850     88.180601     54.565099  0.000287   
6      RBLX     54.540001     40.663500     51.389751     29.937249 -0.186759   
7   0002.hk     65.250000     65.724750     70.963694     60.485806  0.029662   
8   2318.hk     46.150002     38.976500     51.323896     26.629104  0.007395   
9   0005.hk     73.599998     66.692000     74.086792     59.297207  0.007744   
10     AAPL    243.009995    206.714050    254.740343    158.6877

<ipython-input-6-47bb5cc89b42>:114: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  concatenated_df = pd.concat([concatenated_df, last_row_df], ignore_index=True)


In [1]:
import yfinance as yf
import pandas as pd
import requests
import json
import datetime
import gspread
from io import StringIO
from datetime import datetime, timedelta
from google.colab import auth
from gspread_dataframe import set_with_dataframe
from google.auth import default
import contextlib
import sys

def fetch_stock_data(ticker, start_date, end_date):
    # Suppress the console output
    with contextlib.redirect_stdout(None):
        # Fetch historical stock data from Yahoo Finance
        stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
    return stock_data

def calculate_happiness_lines(stock_data, window=20):
    # Calculate the mean and standard deviations
    with contextlib.redirect_stdout(None):
      stock_data['Mean'] = stock_data['Close'].rolling(window=window).mean()
      stock_data['StdDev'] = stock_data['Close'].rolling(window=window).std()

    # Calculate the five lines of happiness
      stock_data['Upper Band'] = stock_data['Mean'] + 2 * stock_data['StdDev']  # Mean + 2 StdDev
      stock_data['Lower Band'] = stock_data['Mean'] - 2 * stock_data['StdDev']  # Mean - 2 StdDev

    return stock_data

def main():

    # Authenticate and create the interface to Sheets
    auth.authenticate_user()
    #gc = gspread.authorize(GoogleCredentials.get_application_default())

    creds, _ = default()
    gc = gspread.authorize(creds)

  # Open the Google Sheet using its name or URL
    spreadsheet = gc.open('Outperform')
    worksheet = spreadsheet.worksheet('Stockcode')

  # Assuming the specific column value you're looking for is "XYZ"
    target_value = "^hsi"

  # Initialize an empty list to store stock symbols
    stocks = []

  # Iterate over each stock symbol
    for stock in worksheet.col_values(1)[1:]:
    # Check if the current stock symbol matches the target value
      if stock == target_value:
          break  # Stop collecting symbols once the target value is found
      else:
          stocks.append(stock)  # Add the stock symbol to the list

    #print(stocks)


    #ticker = 'AAPL'  # Example: Apple Inc.

    # Calculate the current date and the date 500 days before the current date
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=500)).strftime('%Y-%m-%d')

    # Initialize an empty DataFrame to concatenate results
    concatenated_df = pd.DataFrame()

    for ticker in stocks:
        # Fetch data
        stock_data = fetch_stock_data(ticker, start_date, end_date)

        # Calculate happiness lines with a 20-day rolling window
        result = calculate_happiness_lines(stock_data, window=200)

        # Get the last row of the result
        #last_row = result[['Close', 'Mean', 'Upper Band', 'Lower Band']].dropna()

        # Add the ticker symbol to the last row
        #last_row['Ticker'] = ticker

        # Get the last row of the result
        last_row = result[['Close', 'Mean', 'Upper Band', 'Lower Band']].dropna().tail(1)

        # Convert the last row to an array of values
        last_row_values = last_row.values.flatten()

        # Create a DataFrame from the array of values
        last_row_df = pd.DataFrame([last_row_values], columns=['Close', 'Mean', 'Upper Band', 'Lower Band'])

         # Add the ticker symbol to the DataFrame
        last_row_df.insert(0, 'Ticker', ticker)

        # Concatenate the last row DataFrame to the concatenated DataFrame
        concatenated_df = pd.concat([concatenated_df, last_row_df], ignore_index=True)

    # Print the concatenated DataFrame
    print(concatenated_df)


     # Open the 'Current5' worksheet
    worksheet = spreadsheet.worksheet('Yahoo')

    # Set the DataFrame to the worksheet
    set_with_dataframe(worksheet, concatenated_df)

        #Append the last row to the final DataFrame using pd.concat
        #final_df = pd.concat([final_df, last_row])

    # Reset the index of the final DataFrame
    #final_df.reset_index(drop=True, inplace=True)

    # Print the final DataFrame
    #print(final_df)

if __name__ == "__main__":
    main()



     Ticker        Close         Mean   Upper Band   Lower Band
0     GOOGL   171.490005   164.574950   190.570623   138.579278
1         V   316.649994   278.565101   304.385472   252.744729
2      NFLX   897.739990   670.584200   821.402477   519.765923
3      NICE   184.509995   194.011200   255.817126   132.205273
4        SQ    92.779999    71.069100    87.150166    54.988034
5      RBLX    52.160000    40.544750    50.932741    30.156759
6   0002.hk    64.099998    65.720500    70.962371    60.478630
7   2318.hk    45.299999    38.848750    51.161041    26.536459
8   0005.hk    72.550003    66.571500    73.907598    59.235401
9      AAPL   239.589996   206.105100   253.826010   158.384190
10     ORCL   181.410004   141.988950   189.963375    94.014525
11      ZTS   176.809998   177.882350   200.257800   155.506901
12      TXN   201.779999   191.957600   221.666063   162.249137
13      LMT   520.340027   505.232499   619.632775   390.832224
14     NVDA   138.630005   112.390270   

In [9]:
import yfinance as yf

def print_balance_sheet(ticker):
    # Fetch the stock data
    stock = yf.Ticker(ticker)

    # Get the balance sheet data
    balance_sheet = stock.balance_sheet

    # Print the balance sheet
    print(balance_sheet)

# Example usage
print_balance_sheet('AAPL')

                                                      2024-09-30  \
Treasury Shares Number                                       NaN   
Ordinary Shares Number                             15116786000.0   
Share Issued                                       15116786000.0   
Net Debt                                           76686000000.0   
Total Debt                                        106629000000.0   
...                                                          ...   
Cash Cash Equivalents And Short Term Investments   65171000000.0   
Other Short Term Investments                       35228000000.0   
Cash And Cash Equivalents                          29943000000.0   
Cash Equivalents                                    2744000000.0   
Cash Financial                                     27199000000.0   

                                                      2023-09-30  \
Treasury Shares Number                                       0.0   
Ordinary Shares Number                         

In [10]:
import pandas as pd
import yfinance as yf

# Set display options
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None)  # Show full column width
pd.set_option('display.width', None)  # Auto-detect the display width

def print_balance_sheet(ticker):
    # Fetch the stock data
    stock = yf.Ticker(ticker)

    # Get the balance sheet data
    balance_sheet = stock.balance_sheet

    # Print the balance sheet
    print(balance_sheet.to_string())

# Example usage
print_balance_sheet('AAPL')

                                                         2024-09-30      2023-09-30      2022-09-30      2021-09-30      2020-09-30
Treasury Shares Number                                          NaN             0.0             NaN             NaN             NaN
Ordinary Shares Number                                15116786000.0   15550061000.0   15943425000.0   16426786000.0             NaN
Share Issued                                          15116786000.0   15550061000.0   15943425000.0   16426786000.0             NaN
Net Debt                                              76686000000.0   81123000000.0   96423000000.0   89779000000.0             NaN
Total Debt                                           106629000000.0  111088000000.0  132480000000.0  136522000000.0             NaN
Tangible Book Value                                   56950000000.0   62146000000.0   50672000000.0   63090000000.0             NaN
Invested Capital                                     163579000000.0  1732340